# Colab TTS Quality Gate — isolated runtime tests

Qwen3-TTS and Chatterbox-Turbo currently require incompatible `transformers` versions, and Chatterbox pins a different PyTorch version. **Do not install both in the same Colab runtime.**

This notebook therefore uses two isolated sessions:
1. Session A: Qwen3-TTS → save `qwen3_tts_output.wav` + `qwen_report.json`.
2. Restart/delete the Colab runtime.
3. Session B: Chatterbox-Turbo → save `chatterbox_output.wav` + `chatterbox_report.json`.

Both reports survive a runtime restart because they are written under `/content/openmontage-colab/projects/colab-tts-quality/`.

## SESSION A — Qwen3-TTS
Run the next cells in a fresh GPU runtime. Do **not** run the Chatterbox installation in this session.

In [ ]:
import sys, torch
print('Python:',sys.version)
print('Torch:',torch.__version__,'CUDA build:',torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('GPU unavailable. Select a GPU runtime and restart Colab.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

In [ ]:
# Qwen-only environment. Do not install Chatterbox here.
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--only-binary=:all:','numpy==1.26.4'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-cache-dir','qwen-tts==0.1.1','soundfile','scipy'])
print('Qwen dependencies installed.')

In [ ]:
from pathlib import Path
import json,time,traceback,subprocess
import torch,numpy as np,soundfile as sf
from qwen_tts import Qwen3TTSModel
out=Path('/content/openmontage-colab/projects/colab-tts-quality');out.mkdir(parents=True,exist_ok=True)
wav_path=out/'qwen3_tts_output.wav'; report_path=out/'qwen_report.json'
text='A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered. The signal came from a world no telescope had ever seen before. And buried inside that transmission was a message meant for us.'
instruction='Calm cinematic documentary narration. Natural pacing, clear pronunciation, subtle mystery and anticipation, and natural emotional expression.'
report={'model':'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice','status':'NOT_RUN'}
try:
    t=time.time()
    model=Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice',device_map='cuda:0',dtype=torch.bfloat16)
    report['load_seconds']=round(time.time()-t,2)
    t=time.time()
    wavs,sr=model.generate_custom_voice(text=text,language='English',speaker='Ryan',instruct=instruction)
    sf.write(str(wav_path),np.asarray(wavs[0]),int(sr))
    report.update({'generation_seconds':round(time.time()-t,2),'sample_rate':int(sr),'file_size':wav_path.stat().st_size,'status':'REAL_PASS' if wav_path.stat().st_size>1000 else 'REAL_FAIL'})
except Exception as e:
    report.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
report_path.write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
print(json.dumps(report,indent=2,default=str))
print('Saved:',wav_path)


## IMPORTANT — restart before Chatterbox

After Session A finishes successfully:

**Runtime → Disconnect and delete runtime**, then reconnect to a GPU runtime.

Do not run the Qwen cells again in the new runtime. Start at the Chatterbox section below. The Qwen WAV/report remains on the Colab filesystem if the runtime is only restarted; if you delete the runtime, download the Qwen WAV/report first or copy them to Google Drive.


## SESSION B — Chatterbox-Turbo
Run this section only after starting a fresh GPU runtime. Chatterbox's published package pins different `transformers`/PyTorch dependencies from Qwen, so isolation is intentional.

In [ ]:
import sys,torch
print('Python:',sys.version)
print('Torch before install:',torch.__version__,'CUDA build:',torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('GPU unavailable. Select a GPU runtime and restart Colab.')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# Chatterbox-only environment. This package pins its own torch/transformers versions.
# This is why it must be installed in a separate runtime from Qwen.
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--only-binary=:all:','numpy==1.26.4'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-cache-dir','chatterbox-tts==0.1.7','soundfile','scipy'])
print('Chatterbox dependencies installed. A fresh interpreter is already being used for this session.')

In [ ]:
from pathlib import Path
import json,time,traceback
import torch
from chatterbox.tts_turbo import ChatterboxTurboTTS
out=Path('/content/openmontage-colab/projects/colab-tts-quality');out.mkdir(parents=True,exist_ok=True)
wav_path=out/'chatterbox_output.wav'; report_path=out/'chatterbox_report.json'
text='A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered. The signal came from a world no telescope had ever seen before. And buried inside that transmission was a message meant for us.'
report={'model':'ResembleAI/chatterbox-turbo','status':'NOT_RUN'}
try:
    t=time.time(); model=ChatterboxTurboTTS.from_pretrained(device='cuda'); report['load_seconds']=round(time.time()-t,2)
    t=time.time()
    try:
        wav=model.generate(text)
    except AssertionError as first_error:
        # Some checkpoints require an explicit reference voice. If so, ask for one.
        print('Chatterbox needs a reference voice WAV. Upload a short clean English reference below, then rerun this cell.')
        raise RuntimeError('Chatterbox requires audio_prompt_path/prepare_conditionals. Upload a reference WAV and rerun this cell.') from first_error
    import torchaudio
    torchaudio.save(str(wav_path),wav.cpu(),model.sr)
    report.update({'generation_seconds':round(time.time()-t,2),'sample_rate':int(model.sr),'file_size':wav_path.stat().st_size,'status':'REAL_PASS' if wav_path.stat().st_size>1000 else 'REAL_FAIL'})
except Exception as e:
    report.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
report_path.write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
print(json.dumps(report,indent=2,default=str))
print('Saved:',wav_path)

In [ ]:
# Compare reports if both files are available in this runtime.
from pathlib import Path
import json
out=Path('/content/openmontage-colab/projects/colab-tts-quality')
for name in ['qwen_report.json','chatterbox_report.json']:
    p=out/name
    if p.exists(): print(name+'\n'+p.read_text()+'\n')
